# Лабораторна робота №2 — Частина 2

У цій частині аналізується датасет **Individual Household Electric Power Consumption Dataset**.

Потрібно:
1. Завантажити та відкрити датасет.
2. Виконати data cleaning.
3. Сформувати вибірки окремими функціями.
4. Нормувати та стандартизувати датасет.
5. Порахувати коефіцієнти кореляції Пірсона та Спірмена.
6. Провести One Hot Encoding категоріального атрибута.
7. Проаналізувати час виконання процедур за допомогою `timeit`.

**Студент:** Сапронов Анатолій  
**Група:** ФБ-45


In [ ]:
import pandas as pd
import numpy as np
import requests
import zipfile
from pathlib import Path
from timeit import timeit
from scipy.stats import pearsonr, spearmanr

DATA_DIR = Path("data_power")
DATA_DIR.mkdir(exist_ok=True)

ZIP_URL = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
ZIP_PATH = DATA_DIR / "household_power_consumption.zip"
TXT_PATH = DATA_DIR / "household_power_consumption.txt"


## 1. Завантаження датасету

In [ ]:
def download_power_dataset() -> Path:
    if TXT_PATH.exists():
        print("Датасет вже розпакований.")
        return TXT_PATH

    if not ZIP_PATH.exists():
        print("Завантаження датасету...")
        response = requests.get(ZIP_URL, timeout=60)
        response.raise_for_status()
        ZIP_PATH.write_bytes(response.content)

    print("Розпакування архіву...")
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        archive.extractall(DATA_DIR)

    return TXT_PATH


dataset_path = download_power_dataset()
dataset_path


## 2. Data cleaning

У датасеті пропущені значення позначені як `?`.  
Під час очищення:
- `?` замінюються на `NaN`;
- числові колонки переводяться у формат `float`;
- рядки з пропущеними значеннями видаляються;
- створюється колонка `DateTime`.


In [ ]:
def load_and_clean_power_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        sep=";",
        na_values="?",
        low_memory=False
    )

    df["DateTime"] = pd.to_datetime(
        df["Date"] + " " + df["Time"],
        format="%d/%m/%Y %H:%M:%S",
        errors="coerce"
    )

    numeric_cols = [
        "Global_active_power",
        "Global_reactive_power",
        "Voltage",
        "Global_intensity",
        "Sub_metering_1",
        "Sub_metering_2",
        "Sub_metering_3"
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna().reset_index(drop=True)
    return df


power_df = load_and_clean_power_data(dataset_path)
power_df.head()


In [ ]:
power_df.info()


## 3. Функції для формування вибірок

In [ ]:
def select_active_power_over_5kw(df: pd.DataFrame) -> pd.DataFrame:
    return df[df["Global_active_power"] > 5].copy()


def select_current_between_19_and_20(df: pd.DataFrame) -> pd.DataFrame:
    result = df[
        (df["Global_intensity"] >= 19) &
        (df["Global_intensity"] <= 20)
    ].copy()

    result["Sub_metering_sum"] = (
        result["Sub_metering_1"] +
        result["Sub_metering_2"] +
        result["Sub_metering_3"]
    )

    # Пральна машина та холодильник — група 2, бойлер та кондиціонер — група 3.
    result = result[result["Sub_metering_2"] > result["Sub_metering_3"]]
    return result


def sample_500k_and_means(df: pd.DataFrame, random_state: int = 42) -> pd.Series:
    sample_size = min(500_000, len(df))
    sample = df.sample(n=sample_size, random_state=random_state, replace=False)

    cols = ["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]
    return sample[cols].mean()


def select_after_18_power_over_6kw_every_nth(df: pd.DataFrame) -> pd.DataFrame:
    result = df[
        (df["DateTime"].dt.hour >= 18) &
        (df["Global_active_power"] > 6)
    ].copy()

    first_half = result.iloc[:len(result) // 2]
    second_half = result.iloc[len(result) // 2:]

    first_part = first_half.iloc[::3]
    second_part = second_half.iloc[::4]

    return pd.concat([first_part, second_part]).reset_index(drop=True)


## 4. Виконання вибірок

In [ ]:
sample_1 = select_active_power_over_5kw(power_df)
sample_1.head()


In [ ]:
sample_2 = select_current_between_19_and_20(power_df)
sample_2.head()


In [ ]:
sample_3_means = sample_500k_and_means(power_df)
sample_3_means


In [ ]:
sample_4 = select_after_18_power_over_6kw_every_nth(power_df)
sample_4.head()


## 5. Профілювання часу виконання процедур

In [ ]:
time_sample_1 = timeit(lambda: select_active_power_over_5kw(power_df), number=3)
time_sample_2 = timeit(lambda: select_current_between_19_and_20(power_df), number=3)
time_sample_3 = timeit(lambda: sample_500k_and_means(power_df), number=3)
time_sample_4 = timeit(lambda: select_after_18_power_over_6kw_every_nth(power_df), number=3)

timing_results = pd.DataFrame({
    "procedure": [
        "active_power_over_5kw",
        "current_19_20_sub2_gt_sub3",
        "sample_500k_means",
        "after_18_power_over_6kw_every_nth"
    ],
    "time_seconds_for_3_runs": [
        time_sample_1,
        time_sample_2,
        time_sample_3,
        time_sample_4
    ]
})

timing_results


## 6. Нормування та стандартизація

Нормування переводить значення у діапазон `[0; 1]`.  
Стандартизація перетворює дані так, щоб середнє було приблизно `0`, а стандартне відхилення — `1`.


In [ ]:
numeric_cols = [
    "Global_active_power",
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

normalized_df = power_df.copy()
standardized_df = power_df.copy()

for col in numeric_cols:
    min_value = power_df[col].min()
    max_value = power_df[col].max()
    normalized_df[col] = (power_df[col] - min_value) / (max_value - min_value)

    mean_value = power_df[col].mean()
    std_value = power_df[col].std()
    standardized_df[col] = (power_df[col] - mean_value) / std_value

normalized_df[numeric_cols].head()


In [ ]:
standardized_df[numeric_cols].head()


## 7. Коефіцієнти кореляції Пірсона та Спірмена

Для прикладу використано дві числові ознаки:
- `Global_active_power`
- `Global_intensity`


In [ ]:
x = power_df["Global_active_power"]
y = power_df["Global_intensity"]

pearson_corr, pearson_p = pearsonr(x, y)
spearman_corr, spearman_p = spearmanr(x, y)

correlation_results = pd.DataFrame({
    "method": ["Pearson", "Spearman"],
    "correlation": [pearson_corr, spearman_corr],
    "p_value": [pearson_p, spearman_p]
})

correlation_results


## 8. One Hot Encoding категоріального атрибута

У датасеті створимо категоріальну ознаку `day_period`, яка показує частину доби.
Після цього виконаємо One Hot Encoding.


In [ ]:
def get_day_period(hour: int) -> str:
    if 6 <= hour < 12:
        return "morning"
    elif 12 <= hour < 18:
        return "day"
    elif 18 <= hour < 24:
        return "evening"
    return "night"


encoded_df = power_df.copy()
encoded_df["day_period"] = encoded_df["DateTime"].dt.hour.apply(get_day_period)

one_hot = pd.get_dummies(encoded_df["day_period"], prefix="period")
encoded_df = pd.concat([encoded_df, one_hot], axis=1)

encoded_df[["DateTime", "day_period"] + list(one_hot.columns)].head()


## Висновок

У лабораторній роботі було виконано підготовку даних, очищення, формування вибірок за умовами, нормування, стандартизацію, кореляційний аналіз та One Hot Encoding. Також було перевірено час виконання основних процедур за допомогою модуля `timeit`.
